In [1]:
# Ô lệnh do kaggle_run.sh chèn vào — đặt chế độ chạy trước khi notebook đọc biến này.
import os
os.environ['SMOKE_TEST'] = '0'
print('SMOKE_TEST =', os.environ['SMOKE_TEST'], '->', 'thử nhanh' if os.environ['SMOKE_TEST'] == '1' else 'chạy đầy đủ')


SMOKE_TEST = 0 -> chạy đầy đủ


# Lazada Uplift — 09 · Đóng gói bản 30 đặc trưng

Notebook 08 chấm cổng quyết định và **đạt cả ba điều kiện**: hai thứ hạng độc lập đồng ý về tập
30 cột (21/30), điểm của nhánh D nằm trong sàn nhiễu quanh nhánh A (`−0,00037` so với sàn
`±0,00442`), và bootstrap theo cặp không chứng minh được D tệ hơn.

Notebook này huấn luyện DR-Learner 30 cột trên **toàn bộ** `train + val`, rồi xuất bộ artifact
bàn giao **song song** với hậu tố `_k30`.

**Notebook này KHÔNG đè bản 69 cột.** `model.pkl`, `feature_contract.json`, `metadata.json`,
`model_booster.txt` giữ nguyên. Việc đổi bản bàn giao là một thao tác riêng, cần đồng ý riêng —
DE tiêu thụ trực tiếp mấy file đó và thao tác này khó lùi.

**Notebook này KHÔNG nạp `rct_holdout.parquet`.** Bản 30 cột chưa từng được đo trên tập giữ
cuối, và giới hạn đó được ghi thẳng vào `metadata_k30.json` chứ không để trống.

# Phần 1 — Thiết lập và chốt cổng

### Bước 1 — Thư viện và đường dẫn

In [2]:
import os
if os.path.exists('/kaggle'):
    get_ipython().run_line_magic('pip', 'install -q cloudpickle')
else:
    print('Local: dependencies are managed by uv (uv sync --frozen).')

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import json
import time
import glob
import sys
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
import cloudpickle

import warnings
warnings.filterwarnings('ignore')

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    _train_files = glob.glob('/kaggle/input/**/train.parquet', recursive=True)
    if not _train_files:
        raise FileNotFoundError(
            'Không tìm thấy train.parquet trong /kaggle/input. Kiểm tra dataset source.')
    ROOT = '/kaggle/working'
    DATA_DIR = os.path.dirname(_train_files[0])
    ART_DIR = '/kaggle/working/artifacts'
    FIG_DIR = '/kaggle/working/reports/figures'
else:
    ROOT = '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'
    DATA_DIR = os.path.join(ROOT, 'dataset')
    ART_DIR = os.path.join(ROOT, 'artifacts')
    FIG_DIR = os.path.join(ROOT, 'reports', 'figures')

os.makedirs(ART_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)


def tim_file(ten):
    '''Tìm artifact local hoặc trong output của kernel nguồn trên Kaggle.'''
    ung_vien = [os.path.join(ART_DIR, ten)]
    if IS_KAGGLE:
        ung_vien += sorted(glob.glob(f'/kaggle/input/**/{ten}', recursive=True))
    ung_vien = [p for p in ung_vien if os.path.exists(p)]
    if not ung_vien:
        raise FileNotFoundError(
            f'Không tìm thấy {ten}. Chạy notebook 07 và 08 trước, hoặc kiểm tra kernel_sources.')
    if len(ung_vien) > 1:
        print(f'!!! Có {len(ung_vien)} bản {ten}; đang dùng: {ung_vien[0]}')
    return ung_vien[0]


SEED = 42
np.random.seed(SEED)
TRAPZ = getattr(np, 'trapezoid', None) or np.trapz

plt.style.use('seaborn-v0_8-whitegrid')

# Kaggle không có font phủ dấu tiếng Việt nên tiêu đề biểu đồ ra ô vuông. DejaVu Sans đi kèm
# chính matplotlib nên máy nào cũng có, và nó phủ đủ.
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans'] + plt.rcParams['font.sans-serif']

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titleweight'] = 'bold'
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

print('Môi trường :', 'Kaggle' if IS_KAGGLE else 'Local')
print('ART_DIR    :', ART_DIR)
print('rct_holdout: KHÔNG nạp')

Môi trường : Kaggle
ART_DIR    : /kaggle/working/artifacts
rct_holdout: KHÔNG nạp


### Bước 2 — Chế độ chạy

In [4]:
SMOKE_TEST = os.environ.get('SMOKE_TEST', '1').lower() not in ('0', 'false', 'no')

N_FIT = 50_000 if SMOKE_TEST else None      # None = dùng toàn bộ train + val
RUN_MODE = 'smoke_test' if SMOKE_TEST else 'full'

print('=' * 70)
print(f'  09 — ĐÓNG GÓI BẢN 30 CỘT   |   CHẾ ĐỘ: {RUN_MODE.upper()}')
print(f'  Số dòng huấn luyện: {"toàn bộ train + val" if N_FIT is None else f"{N_FIT:,}"}')
if SMOKE_TEST:
    print('  >>> CHỈ ĐỂ KIỂM TRA CODE — artifact sinh ra KHÔNG dùng để bàn giao <<<')
print('=' * 70)

  09 — ĐÓNG GÓI BẢN 30 CỘT   |   CHẾ ĐỘ: FULL
  Số dòng huấn luyện: toàn bộ train + val


### Bước 3 — Chốt cổng: notebook này chỉ chạy nếu notebook 08 đã ĐẠT

Đây là chốt cổng quan trọng nhất của notebook. Nếu `so_sanh_k30.json` không phải bản `full`,
hoặc cổng không đạt, ô lệnh dưới dừng ngay — không có đường nào đóng gói được một bản mà bằng
chứng chưa qua cổng.

In [5]:
so_sanh = json.load(open(tim_file('so_sanh_k30.json'), encoding='utf-8'))
cong = so_sanh['cong_quyet_dinh']

assert so_sanh['run_mode'] == 'full', (
    f'so_sanh_k30.json đang là kết quả {so_sanh["run_mode"]!r}. '
    'Chạy notebook 08 ở chế độ full trước.')
assert cong['ket_qua']['dat'], (
    'Cổng quyết định KHÔNG đạt: ' + json.dumps(cong['ket_qua'], ensure_ascii=False) +
    ' — giữ nguyên bản 69 cột, không đóng gói bản rút gọn.')

# Băm lại định nghĩa cổng và so với mã băm notebook 08 đã chốt trước khi có số.
ma_bam_lai = hashlib.sha256(
    json.dumps(cong['dinh_nghia'], sort_keys=True, ensure_ascii=False).encode('utf-8')
).hexdigest()[:16]
assert ma_bam_lai == cong['ma_bam'], (
    f'Định nghĩa cổng trong so_sanh_k30.json đã bị sửa ({cong["ma_bam"]} -> {ma_bam_lai}). '
    'Chạy lại notebook 08.')

print('🔒 Cổng quyết định ĐẠT, mã băm khớp:', cong['ma_bam'])
for k, v in cong['ket_qua'].items():
    print(f'   {k}: {v}')

NHANH_D = next(n for n in so_sanh['nhanh'] if n['ma'] == 'D')
NHANH_A = next(n for n in so_sanh['nhanh'] if n['ma'] == 'A')
SAN_NHIEU = so_sanh['san_nhieu']
print()
print(f'   nhánh A (69 cột): Qini {NHANH_A["Qini_TB"]:+.5f}')
print(f'   nhánh D (30 cột): Qini {NHANH_D["Qini_TB"]:+.5f}  '
      f'(lệch {NHANH_D["lech_so_voi_A"]:+.5f}, sàn nhiễu ±{SAN_NHIEU})')

🔒 Cổng quyết định ĐẠT, mã băm khớp: a4f5c2ff6ae084f6
   dieu_kien_1: True
   dieu_kien_2: True
   dieu_kien_3: True
   dat: True

   nhánh A (69 cột): Qini +0.02566
   nhánh D (30 cột): Qini +0.02529  (lệch -0.00037, sàn nhiễu ±0.00442)


### Bước 4 — Nạp tham số, đặc trưng và dữ liệu

In [6]:
tuning69 = json.load(open(tim_file('best_params.json'), encoding='utf-8'))
E_ALPHA = float(tuning69['xu_ly_overlap']['propensity_alpha'])
E_CLIP = (E_ALPHA, 1 - E_ALPHA)

tuning30 = json.load(open(tim_file('best_params_k30.json'), encoding='utf-8'))
P_DR30 = tuning30['best_params']['DRLearner']['params']

xh_val = json.load(open(tim_file('xep_hang_val.json'), encoding='utf-8'))
FEAT_CHOT = xh_val['dac_trung_o_muc_chot']
HANG_VAL = xh_val['hang']

select_meta = json.load(open(tim_file('04b_select_meta.json'), encoding='utf-8'))
HANG_SELECT = select_meta['permutation_importance']['hang']
HANG_GAIN = select_meta['gain_importance']['hang']

finfo = json.load(open(tim_file('feature_info.json'), encoding='utf-8'))
FEATS = finfo['tat_ca_dac_trung']
N_FEAT = len(FEATS)
K_CHOT = len(FEAT_CHOT)

assert tuning30['run_mode'] == 'full', 'best_params_k30.json phải là bản full'
assert xh_val['run_mode'] == 'full', 'xep_hang_val.json phải là bản full'
assert tuning30['cau_hinh']['feature_names'] == FEAT_CHOT, \
    'best_params_k30.json tinh chỉnh trên tập cột khác xep_hang_val.json'
assert NHANH_D['dac_trung'] == FEAT_CHOT, \
    'Nhánh D của notebook 08 dùng tập cột khác — không đóng gói bản chưa được chấm'
assert FEAT_CHOT == [f for f in FEATS if f in set(FEAT_CHOT)], \
    'FEAT_CHOT phải giữ thứ tự cột gốc'

train = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'))
val = pd.read_parquet(os.path.join(DATA_DIR, 'val.parquet'))
rct_sel = pd.read_parquet(os.path.join(DATA_DIR, 'rct_select.parquet'))
full = pd.concat([train, val], ignore_index=True)

if N_FIT is not None and N_FIT < len(full):
    rng_mau = np.random.RandomState(SEED)
    strata = full['is_treat'].values * 2 + full['label'].values
    _idx = []
    for s in np.unique(strata):
        pos = np.where(strata == s)[0]
        n = max(1, int(round(N_FIT * len(pos) / len(full))))
        _idx.append(rng_mau.choice(pos, size=min(n, len(pos)), replace=False))
    full = full.iloc[np.sort(np.concatenate(_idx))].reset_index(drop=True)

IDX_CHOT = [i for i, f in enumerate(FEATS) if f in set(FEAT_CHOT)]
X_train = full[FEAT_CHOT].values.astype(np.float64)
W_train, Y_train = full['is_treat'].values, full['label'].values
X_sel_chot = rct_sel[FEAT_CHOT].values.astype(np.float64)

RUN_ID = f'{RUN_MODE}-{datetime.now().strftime("%Y%m%d-%H%M%S")}'

print(f'Huấn luyện trên : {len(full):>8,} dòng | treat {W_train.mean()*100:.2f}% '
      f'| mua {Y_train.mean()*100:.2f}%')
print(f'Đặc trưng       : {K_CHOT}/{N_FEAT} cột')
print(f'Siêu tham số    : {P_DR30}')
print(f'Ngưỡng clip ê   : {E_CLIP}')
print(f'run_id          : {RUN_ID}')
print()
print('30 cột (thứ tự cột gốc = thứ tự đưa vào mô hình):')
print('  ' + ', '.join(FEAT_CHOT))

Huấn luyện trên :  926,669 dòng | treat 22.17% | mua 1.99%
Đặc trưng       : 30/69 cột
Siêu tham số    : {'n_estimators': 300, 'learning_rate': 0.01459007452373112, 'max_depth': 4, 'min_child_samples': 46}
Ngưỡng clip ê   : (0.071, 0.929)
run_id          : full-20260820-093115

30 cột (thứ tự cột gốc = thứ tự đưa vào mô hình):
  f0, f1, f3, f4, f5, f6, f7, f8, f9, f10, f13, f16, f17, f20, f21, f22, f23, f25, f26, f27, f28, f29, f35, f38, f42, f52, f60, f68, f80, f82


# Phần 2 — Huấn luyện bản bàn giao

### Bước 5 — DR-Learner

Bản sao đúng của class ở notebook 03. Phải giống hệt: `cloudpickle` đóng gói class kèm code
object, nên class ở đây chính là thứ phía DE nạp lại.

In [7]:
def fit_nuisance(X, W, Y, n_folds=5, seed=SEED):
    '''Ước lượng ba mô hình phụ bằng cross-fitting. Trả về ê THÔ, chưa cắt.'''
    n = len(X)
    e_raw, mu0_hat, mu1_hat = np.zeros(n), np.zeros(n), np.zeros(n)
    strata = W * 2 + Y
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    nuisance_params = dict(n_estimators=200, learning_rate=0.05, max_depth=5,
                           min_child_samples=50, random_state=seed, verbose=-1)

    for tr, te in skf.split(X, strata):
        e_raw[te] = lgb.LGBMClassifier(**nuisance_params).fit(
            X[tr], W[tr]).predict_proba(X[te])[:, 1]
        mu0_hat[te] = lgb.LGBMClassifier(**nuisance_params).fit(
            X[tr][W[tr] == 0], Y[tr][W[tr] == 0]).predict_proba(X[te])[:, 1]
        mu1_hat[te] = lgb.LGBMClassifier(**nuisance_params).fit(
            X[tr][W[tr] == 1], Y[tr][W[tr] == 1]).predict_proba(X[te])[:, 1]

    return e_raw, mu0_hat, mu1_hat


def dr_from_nuisance(W, Y, e_raw, mu0_hat, mu1_hat, clip):
    e_hat = np.clip(e_raw, *clip)
    dr = ((mu1_hat - mu0_hat)
          + W * (Y - mu1_hat) / e_hat
          - (1 - W) * (Y - mu0_hat) / (1 - e_hat))
    return dr, e_hat


def compute_dr_scores(X, W, Y, n_folds=5, seed=SEED, clip=None):
    clip = E_CLIP if clip is None else clip
    e_raw, mu0_hat, mu1_hat = fit_nuisance(X, W, Y, n_folds=n_folds, seed=seed)
    dr, e_hat = dr_from_nuisance(W, Y, e_raw, mu0_hat, mu1_hat, clip)
    return dr, e_hat, mu0_hat, mu1_hat


class DRLearner:
    '''Kennedy 2020 — hồi quy pseudo-outcome doubly robust.'''
    def __init__(self, n_folds=5, **params):
        self.n_folds = n_folds
        self.params = dict(params, random_state=SEED, verbose=-1)

    def fit(self, X, W, Y):
        dr_train, *_ = compute_dr_scores(X, W, Y, n_folds=self.n_folds)
        self.model = lgb.LGBMRegressor(**self.params).fit(X, dr_train)
        return self

    def predict_cate(self, X):
        return self.model.predict(X)


print('Đã định nghĩa DRLearner')

Đã định nghĩa DRLearner


### Bước 6 — Huấn luyện

In [8]:
t0 = time.time()
MO_HINH = DRLearner(**P_DR30).fit(X_train, W_train, Y_train)
GIAY_HUAN_LUYEN = time.time() - t0

cate_sel = MO_HINH.predict_cate(X_sel_chot)

print(f'Huấn luyện xong sau {GIAY_HUAN_LUYEN:.1f} giây ({GIAY_HUAN_LUYEN/60:.1f} phút)')
print(f'CATE trên rct_select: trung bình {cate_sel.mean():+.6f} | '
      f'sd {cate_sel.std():.6f} | [{cate_sel.min():+.6f} .. {cate_sel.max():+.6f}]')

Huấn luyện xong sau 174.8 giây (2.9 phút)
CATE trên rct_select: trung bình +0.009566 | sd 0.012627 | [-0.245800 .. +0.336671]


# Phần 3 — Xuất bộ artifact `_k30`

### Bước 7 — `model_k30.pkl`

`cloudpickle` đóng gói class kèm cả code object, mà cấu trúc code object **không tương thích
giữa các bản Python**. Bản Python đóng gói được ghi vào `metadata_k30.json` để phía nạp biết
phải khớp.

In [9]:
MODEL_PATH = os.path.join(ART_DIR, 'model_k30.pkl')
with open(MODEL_PATH, 'wb') as f:
    cloudpickle.dump(MO_HINH, f)

PYTHON_DONG_GOI = f'{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}'

print(f'{MODEL_PATH} — DRLearner, {K_CHOT} đặc trưng, '
      f'{os.path.getsize(MODEL_PATH)/1e6:.2f} MB')
print(f'  cloudpickle {cloudpickle.__version__} | Python đóng gói {PYTHON_DONG_GOI}')
print('  KHÔNG đè model.pkl — bản 69 cột giữ nguyên.')

/kaggle/working/artifacts/model_k30.pkl — DRLearner, 30 đặc trưng, 0.45 MB
  cloudpickle 3.1.2 | Python đóng gói 3.12.13
  KHÔNG đè model.pkl — bản 69 cột giữ nguyên.


### Bước 8 — `model_booster_k30.txt` — bản không phụ thuộc phiên bản Python

Đối chiếu lại với `model_k30.pkl` trên 3.000 dòng: LightGBM tất định tới từng bit nên đòi
lệch **đúng bằng 0**, không phải "đủ nhỏ".

In [10]:
BOOSTER_PATH = os.path.join(ART_DIR, 'model_booster_k30.txt')
MO_HINH.model.booster_.save_model(BOOSTER_PATH)

booster = lgb.Booster(model_file=BOOSTER_PATH)
assert booster.num_feature() == K_CHOT, \
    f'booster nhận {booster.num_feature()} cột nhưng contract có {K_CHOT}'

X_kiem = X_sel_chot[:3000]
LECH_BOOSTER = float(np.abs(booster.predict(X_kiem) - MO_HINH.predict_cate(X_kiem)).max())
assert LECH_BOOSTER == 0.0, \
    f'Bản text lệch {LECH_BOOSTER:.3e} so với model_k30.pkl — không dùng thay được'

BOOSTER_INFO = {
    'files': [{'file': 'model_booster_k30.txt',
               'so_cot_dau_vao': int(booster.num_feature()),
               'so_cay': int(booster.num_trees()),
               'mo_ta': ('LGBMRegressor tầng cuối — đầu vào đúng các cột trong '
                         'feature_contract_k30.json, đầu ra là CATE')}],
    'so_cot_contract': int(K_CHOT),
    'cot_bo_sung': [],
    'cach_ghep_cate': 'booster.predict(X)',
    'lech_so_voi_model_pkl': LECH_BOOSTER,
}

print(f'{BOOSTER_PATH} — {booster.num_feature()} cột đầu vào, {booster.num_trees()} cây, '
      f'{os.path.getsize(BOOSTER_PATH)/1e6:.2f} MB')
print(f'Đối chiếu 3.000 dự đoán với model_k30.pkl: lệch {LECH_BOOSTER:.3e}  ->  trùng khít')

/kaggle/working/artifacts/model_booster_k30.txt — 30 cột đầu vào, 300 cây, 0.45 MB
Đối chiếu 3.000 dự đoán với model_k30.pkl: lệch 0.000e+00  ->  trùng khít


### Bước 9 — `feature_contract_k30.json`

Giữ **thứ tự cột gốc**, không phải thứ tự theo độ quan trọng. Sai chỗ này thì phía DE dựng
vector sai mà mô hình vẫn trả về số, không báo lỗi.

Contract mang **cả hai** thứ hạng: `hang_permutation` là hạng đo trên Val — thứ hạng đã chọn ra
tập cột này — và `hang_permutation_rct_select` là hạng của 04b, để đối chiếu.

In [11]:
muc = []
for vi_tri, f in enumerate(FEAT_CHOT):
    col = full[f]
    muc.append({
        'ten': f,
        'thu_tu_dua_vao_mo_hinh': vi_tri,
        'nguon': 'goc',
        'kieu': 'nhi_phan' if col.nunique() == 2 else 'lien_tuc',
        'gia_tri_mac_dinh': float(col.median()),
        'khoang': [float(col.min()), float(col.max())],
        'hang_permutation': HANG_VAL.get(f),
        'hang_permutation_rct_select': HANG_SELECT.get(f),
        'hang_gain': HANG_GAIN.get(f),
        'tinh_online': None,
    })

contract = {
    'phien_ban': datetime.now().strftime('%Y%m%d'),
    'mo_hinh': 'DRLearner',
    'file_mo_hinh': 'model_k30.pkl',
    'file_khong_phu_thuoc_python': ['model_booster_k30.txt'],
    'so_cot_mo_hinh_nhan_vao': len(muc),
    'nguon_dac_trung': ('Toàn bộ lấy từ feature store. Pipeline này không có đặc trưng dẫn xuất '
                        'nên AI Service không phải tính công thức nào lúc phục vụ.'),
    'cach_dung': (f'Dựng vector đủ {len(muc)} cột theo thu_tu_dua_vao_mo_hinh rồi gọi '
                  f'model.predict_cate(X). Cột thiếu dữ liệu thì điền gia_tri_mac_dinh.'),
    'canh_bao_thu_tu': ('Sai thứ tự cột sẽ cho dự đoán sai mà KHÔNG báo lỗi. '
                        'Luôn dựng vector theo thu_tu_dua_vao_mo_hinh.'),
    'cach_chon_so_dac_trung': {
        'ket_qua': f'{len(muc)}/{N_FEAT}',
        'ly_do': ('k = 30 thừa kế từ notebook 05; thứ hạng đặc trưng dựng lại trên Val ở '
                  'notebook 07 để rct_select không vừa chọn vừa chấm'),
        'cach_do': ('permutation importance trên vùng overlap của Val (notebook 07), rồi so '
                    '4 nhánh nhiều seed trên rct_select kèm bootstrap theo cặp (notebook 08)'),
        'tieu_chi': [d['mo_ta'] for d in
                     (cong['dinh_nghia']['dieu_kien_1'], cong['dinh_nghia']['dieu_kien_2'],
                      cong['dinh_nghia']['dieu_kien_3'])],
        'tap_dung_de_chot': 'Val (xếp hạng) + rct_select (chấm điểm)',
        'thu_tu_bo_cot': 'permutation importance đo trên vùng overlap của Val (notebook 07)',
        'ma_bam_cong': cong['ma_bam'],
    },
    'gia_tri_mac_dinh_lay_tu': 'trung vị trên train + val',
    'gia_tri_mac_dinh_dung_khi_nao': 'khách hàng mới chưa có dữ liệu trong feature store',
    'ghi_chu_online_offline': ('Dữ liệu đã ẩn danh nên chưa xác định được đặc trưng nào tính '
                               'online, đặc trưng nào tính offline theo cửa sổ 7/30 ngày. '
                               'DE điền trường tinh_online sau khi đối chiếu bảng gốc.'),
    'golden_predictions': 'golden_predictions_k30.csv — xem Bước 10',
    'gioi_han': ('Bản này CHƯA TỪNG được đo trên rct_holdout. Bằng chứng ngoài mẫu duy nhất là '
                 'rct_select, với thứ hạng đặc trưng dẫn xuất từ Val. Xem metadata_k30.json.'),
    'dac_trung': muc,
}

assert [m['ten'] for m in muc] == FEAT_CHOT
assert [m['thu_tu_dua_vao_mo_hinh'] for m in muc] == list(range(len(muc)))
assert not any(m['ten'].startswith('fe_') for m in muc)
assert contract['so_cot_mo_hinh_nhan_vao'] == K_CHOT
assert all(np.isfinite(m['gia_tri_mac_dinh']) for m in muc)
for t in BOOSTER_INFO['files']:
    assert t['so_cot_dau_vao'] == len(muc)

CONTRACT_PATH = os.path.join(ART_DIR, 'feature_contract_k30.json')
with open(CONTRACT_PATH, 'w', encoding='utf-8') as f:
    json.dump(contract, f, ensure_ascii=False, indent=2)

print(f'Đã ghi {CONTRACT_PATH} — {len(muc)} cột, toàn bộ là cột gốc f*')
display(pd.DataFrame(muc)[['thu_tu_dua_vao_mo_hinh', 'ten', 'kieu', 'gia_tri_mac_dinh',
                           'hang_permutation', 'hang_permutation_rct_select']].head(10))

Đã ghi /kaggle/working/artifacts/feature_contract_k30.json — 30 cột, toàn bộ là cột gốc f*


,thu_tu_dua_vao_mo_hinh,ten,kieu,gia_tri_mac_dinh,hang_permutation,hang_permutation_rct_select
0,0,f0,lien_tuc,2.000000,12,8
1,1,f1,lien_tuc,172.000000,9,13
2,2,f3,lien_tuc,1.098612,3,12
3,3,f4,lien_tuc,1.500000,8,27
4,4,f5,lien_tuc,0.000000,13,24
5,5,f6,lien_tuc,0.000000,17,60
6,6,f7,lien_tuc,2.000000,30,46
7,7,f8,lien_tuc,1.000000,24,22
8,8,f9,lien_tuc,2.028148,28,18
9,9,f10,lien_tuc,1.547562,7,21


### Bước 10 — `golden_predictions_k30.csv`

1.000 dự đoán mẫu để DE đối chiếu. Sắp theo `data_id` nên tập này không phụ thuộc thứ tự dòng
trong file parquet.

In [12]:
GOLDEN_N = 1_000
TOL_PARITY = 0.0          # LightGBM tất định tới từng bit

thu_tu = np.argsort(rct_sel['data_id'].astype(str).values, kind='stable')[:GOLDEN_N]
X_golden = X_sel_chot[thu_tu]
cate_golden = np.asarray(MO_HINH.predict_cate(X_golden), dtype=np.float64)

golden = pd.DataFrame({'data_id': rct_sel['data_id'].values[thu_tu], 'cate': cate_golden})
GOLDEN_PATH = os.path.join(ART_DIR, 'golden_predictions_k30.csv')
golden.to_csv(GOLDEN_PATH, index=False, float_format='%.17g')

# float_precision='round_trip' là bắt buộc — bộ phân tích mặc định của pandas nhanh hơn nhưng
# có thể lệch 1 ULP, đủ để phá phép so bằng tuyệt đối ở Bước 12.
golden_lai = pd.read_csv(GOLDEN_PATH, float_precision='round_trip')
assert np.array_equal(golden_lai['cate'].values, cate_golden), \
    'File CSV không tái lập đúng giá trị float64'

GOLDEN_HASH = hashlib.sha256(open(GOLDEN_PATH, 'rb').read()).hexdigest()[:16]

print(f'Đã ghi {GOLDEN_PATH}')
print(f'  {len(golden):,} dòng từ rct_select | mã băm {GOLDEN_HASH}')
print(f'  CATE: trung bình {cate_golden.mean():+.6f} | '
      f'nhỏ nhất {cate_golden.min():+.6f} | lớn nhất {cate_golden.max():+.6f}')
display(golden.head(5))

Đã ghi /kaggle/working/artifacts/golden_predictions_k30.csv
  1,000 dòng từ rct_select | mã băm ae56e6fdd558df1b
  CATE: trung bình +0.009592 | nhỏ nhất -0.245800 | lớn nhất +0.298289


,data_id,cate
0,test_0,0.010519
1,test_1,0.023974
2,test_100,0.017582
3,test_10000,0.011126
4,test_100002,0.004789


### Bước 11 — `metadata_k30.json`

`so_dac_trung_day_du` giữ nguyên `69` và `dac_trung_day_du` giữ nguyên toàn bộ danh sách của
`feature_info.json` — bộ test hợp đồng chốt hai trường này, và chúng nói về **không gian đặc
trưng của dự án**, không phải về số cột mô hình nhận vào.

Trường `gioi_han` là chỗ ghi thẳng cái giá của bản này: nó chưa từng được đo trên `rct_holdout`.

In [13]:
metadata = {
    'ten_mo_hinh': 'DRLearner',
    'ngay_dong_goi': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'run_mode': RUN_MODE,
    'run_id': RUN_ID,
    'seed': SEED,
    'sieu_tham_so': P_DR30,

    'chuoi_notebook': {
        'nguon_xep_hang_dac_trung': 'notebook 07 — permutation importance trên vùng overlap Val',
        'nguon_sieu_tham_so': 'notebook 07 — Optuna 12 trial, DR-AUUC trên Val',
        'nguon_bang_chung': 'notebook 08 — 4 nhánh × 5 seed trên rct_select, bootstrap theo cặp',
        'run_mode': {'07': tuning30['run_mode'], '08': so_sanh['run_mode'], '09': RUN_MODE},
    },

    'dac_trung': FEAT_CHOT,
    'so_dac_trung': int(K_CHOT),
    'dac_trung_day_du': FEATS,
    'so_dac_trung_day_du': int(N_FEAT),
    'khong_co_dac_trung_dan_xuat': True,

    'cong_quyet_dinh': {
        'ma_bam': cong['ma_bam'],
        'ket_qua': cong['ket_qua'],
        'dinh_nghia': cong['dinh_nghia'],
    },

    'bang_chung_rct_select': {
        'san_nhieu': SAN_NHIEU,
        'nhanh': [{k: n[k] for k in ('ma', 'nhan', 'so_cot', 'Qini_TB', 'sd_seed',
                                     'lech_so_voi_A', 'qini_tung_seed')}
                  for n in so_sanh['nhanh']],
        'bootstrap_theo_cap': so_sanh['bootstrap_theo_cap'],
        'doi_chieu_hai_xep_hang': so_sanh['doi_chieu_hai_xep_hang'],
        'phat_bieu_toi_da': ('Bản 30 cột KHÔNG TỆ HƠN bản 69 cột trong phạm vi phân giải của '
                             'phép đo. KHÔNG được phát biểu là "tốt hơn".'),
    },

    'gioi_han': {
        'chua_do_tren_rct_holdout': True,
        'giai_thich': ('rct_holdout đã được mở đúng một lần cho bản 69 cột ở notebook 04c. '
                       'Bản 30 cột này KHÔNG được đo lại ở đó — đo lại là lần mở thứ hai, '
                       'phá quy tắc "một lần chọn, một lần mở holdout" của dự án.'),
        'bang_chung_ngoai_mau_duy_nhat': 'rct_select',
        'thien_vi_con_lai': ('Thứ hạng đặc trưng dẫn xuất từ Val nên rct_select là tập giữ '
                             'ngoài thật cho phép so 30-vs-69. Nhưng k = 30 thì thừa kế từ '
                             'notebook 05, nơi k được chọn trên chính rct_select.'),
        'bien_do_chung_minh_duoc': so_sanh['bootstrap_theo_cap']['bien_do_chung_minh_duoc'],
        'doc_bien_do_the_nao': ('Biên độ "không kém hơn" chặt nhất mà rct_select chứng nhận '
                                'được xấp xỉ bằng chính giá trị Qini, nên điều kiện 3 của cổng '
                                'gần như không mang thông tin. Quyết định thực chất dựa vào '
                                'điều kiện 1 và 2.'),
        'so_sanh_voi_ban_69_cot': ('Bản 69 cột CÓ số trên rct_holdout (Qini 0,0161, KTC chứa 0, '
                                   'xếp 6/6). Đổi sang bản 30 cột nghĩa là mô hình đang chạy '
                                   'production không còn con số nào trên tập giữ cuối.'),
    },

    'file_ban_giao': {
        'model_k30.pkl': f'DRLearner trên {K_CHOT} đặc trưng — bản khớp feature_contract_k30.json',
        'model_booster_k30.txt': ('LGBMRegressor tầng cuối — đầu vào đúng các cột trong '
                                  'feature_contract_k30.json, đầu ra là CATE'),
        'feature_contract_k30.json': f'{K_CHOT} cột, đúng thứ tự đưa vào mô hình',
        'golden_predictions_k30.csv': f'{GOLDEN_N} dự đoán mẫu để DE đối chiếu',
    },
    'khong_de_len_ban_69_cot': ['model.pkl', 'model_booster.txt', 'feature_contract.json',
                                'metadata.json', 'golden_predictions.csv'],

    'booster': BOOSTER_INFO,
    'golden_predictions': {
        'file': 'golden_predictions_k30.csv',
        'nguon': f'rct_select, {GOLDEN_N} data_id nhỏ nhất theo thứ tự chuỗi',
        'so_dong': int(len(golden)),
        'ma_bam': GOLDEN_HASH,
        'dung_sai': TOL_PARITY,
        'cach_kiem': ('dựng vector theo feature_contract_k30.json cho từng data_id, gọi '
                      'predict_cate, so với cột cate — chênh lệch phải nằm trong dung_sai'),
    },
    'serialization': {
        'writer': f'cloudpickle=={cloudpickle.__version__}',
        'reader': 'pickle.load',
        'python_dong_goi': PYTHON_DONG_GOI,
        'python_yeu_cau_khi_nap': ('phải khớp python_dong_goi — cloudpickle không tương thích '
                                   'chéo bản Python'),
        'ban_khong_phu_thuoc_python': ['model_booster_k30.txt'],
    },
    'du_lieu_huan_luyen': {
        'nguon': 'train.parquet + val.parquet',
        'so_dong': int(len(full)),
        'giay_huan_luyen': round(GIAY_HUAN_LUYEN, 1),
        'propensity_clip': [E_CLIP[0], E_CLIP[1]],
    },
}

assert metadata['so_dac_trung_day_du'] == N_FEAT
assert metadata['dac_trung_day_du'] == FEATS
assert metadata['dac_trung'] == [m['ten'] for m in muc]

METADATA_PATH = os.path.join(ART_DIR, 'metadata_k30.json')
with open(METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f'Đã ghi {METADATA_PATH}')
print(f'  dac_trung          : {K_CHOT} cột')
print(f'  dac_trung_day_du   : {N_FEAT} cột (không gian đặc trưng của dự án, giữ nguyên)')
print(f'  gioi_han           : chưa đo trên rct_holdout — đã ghi rõ')

Đã ghi /kaggle/working/artifacts/metadata_k30.json
  dac_trung          : 30 cột
  dac_trung_day_du   : 69 cột (không gian đặc trưng của dự án, giữ nguyên)
  gioi_han           : chưa đo trên rct_holdout — đã ghi rõ


# Phần 4 — Nghiệm thu

### Bước 12 — Nạp lại từ file và đối chiếu

Nạp lại bằng `pickle.load` thuần (đúng cách phía DE làm), dựng lại vector từ contract, rồi so
với `golden_predictions_k30.csv`. Đây là phép kiểm cuối: nếu bước này qua thì phía DE cầm bốn
file kia là dùng được.

In [14]:
import pickle

with open(MODEL_PATH, 'rb') as f:
    mo_hinh_lai = pickle.load(f)

ct = json.load(open(CONTRACT_PATH, encoding='utf-8'))
ten_cot = [d['ten'] for d in sorted(ct['dac_trung'], key=lambda d: d['thu_tu_dua_vao_mo_hinh'])]
assert ten_cot == FEAT_CHOT

# Dựng vector đúng như AI Service sẽ làm: tra theo data_id, xếp cột theo contract.
gp = pd.read_csv(GOLDEN_PATH, float_precision='round_trip')
tra = rct_sel.set_index('data_id').loc[gp['data_id'].values, ten_cot]
X_kiem_lai = tra.values.astype(np.float64)

lech_pkl = float(np.abs(mo_hinh_lai.predict_cate(X_kiem_lai) - gp['cate'].values).max())
lech_bst = float(np.abs(lgb.Booster(model_file=BOOSTER_PATH).predict(X_kiem_lai)
                        - gp['cate'].values).max())

nghiem_thu = pd.DataFrame([
    {'Phép kiểm': 'model_k30.pkl nạp lại bằng pickle.load',
     'Kết quả': type(mo_hinh_lai).__name__, 'Đạt': type(mo_hinh_lai).__name__ == 'DRLearner'},
    {'Phép kiểm': 'contract có đúng 30 cột, thứ tự liên tục từ 0',
     'Kết quả': f'{ct["so_cot_mo_hinh_nhan_vao"]} cột',
     'Đạt': ct['so_cot_mo_hinh_nhan_vao'] == K_CHOT},
    {'Phép kiểm': 'model_k30.pkl vs golden (dựng vector theo contract)',
     'Kết quả': f'{lech_pkl:.3e}', 'Đạt': lech_pkl <= TOL_PARITY},
    {'Phép kiểm': 'model_booster_k30.txt vs golden',
     'Kết quả': f'{lech_bst:.3e}', 'Đạt': lech_bst <= TOL_PARITY},
    {'Phép kiểm': 'bản 69 cột còn nguyên',
     'Kết quả': 'model.pkl, feature_contract.json, metadata.json',
     'Đạt': all(os.path.exists(os.path.join(ART_DIR, t))
                for t in ('model.pkl', 'feature_contract.json', 'metadata.json'))},
])
display(nghiem_thu)
assert nghiem_thu['Đạt'].all(), 'Có phép nghiệm thu không đạt — không bàn giao bản này'
print('✅ Bộ artifact _k30 nghiệm thu đạt.')

,Phép kiểm,Kết quả,Đạt
0,model_k30.pkl nạp lại bằng pickle.load,DRLearner,True
1,"contract có đúng 30 cột, thứ tự liên tục từ 0",30 cột,True
2,model_k30.pkl vs golden (dựng vector theo cont...,0.000e+00,True
3,model_booster_k30.txt vs golden,0.000e+00,True
4,bản 69 cột còn nguyên,"model.pkl, feature_contract.json, metadata.json",False


AssertionError: Có phép nghiệm thu không đạt — không bàn giao bản này

### Bước 13 — Bảng tổng hợp

In [ ]:
tong_hop = pd.DataFrame([
    {'Hạng mục': 'Mô hình', 'Bản 69 cột (đang bàn giao)': 'DRLearner',
     'Bản 30 cột (mới đóng gói)': 'DRLearner'},
    {'Hạng mục': 'Số cột mô hình nhận vào', 'Bản 69 cột (đang bàn giao)': f'{N_FEAT}',
     'Bản 30 cột (mới đóng gói)': f'{K_CHOT}'},
    {'Hạng mục': 'Nguồn xếp hạng đặc trưng', 'Bản 69 cột (đang bàn giao)': '—',
     'Bản 30 cột (mới đóng gói)': 'Val (notebook 07)'},
    {'Hạng mục': 'Qini trên rct_select (5 seed)',
     'Bản 69 cột (đang bàn giao)': f'{NHANH_A["Qini_TB"]:+.5f}',
     'Bản 30 cột (mới đóng gói)': f'{NHANH_D["Qini_TB"]:+.5f}'},
    {'Hạng mục': 'Qini trên rct_holdout',
     'Bản 69 cột (đang bàn giao)': '+0,01612 (KTC chứa 0, xếp 6/6)',
     'Bản 30 cột (mới đóng gói)': 'CHƯA ĐO — và sẽ không đo'},
    {'Hạng mục': 'File mô hình', 'Bản 69 cột (đang bàn giao)': 'model.pkl',
     'Bản 30 cột (mới đóng gói)': 'model_k30.pkl'},
])
display(tong_hop)

print('=' * 78)
print('  ĐÃ ĐÓNG GÓI BẢN 30 CỘT — nhưng CHƯA đổi bản bàn giao.')
print('=' * 78)
print('  Năm file mới, không đè lên gì:')
for t in ('model_k30.pkl', 'model_booster_k30.txt', 'feature_contract_k30.json',
          'golden_predictions_k30.csv', 'metadata_k30.json'):
    print(f'    · {t}')
print()
print('  Muốn đổi thật thì còn ba việc, làm theo thứ tự:')
print('    1. Đè 5 file bàn giao bằng bản _k30 — thao tác riêng, cần đồng ý riêng.')
print('    2. Chạy notebook 10: ngưỡng hiện tại tính từ mô hình cũ, đổi mô hình là hết hiệu lực.')
print('    3. Báo DE đồng bộ lại — demo_ba/artifact_check.py đối chiếu với repo nhung-lala.')
if SMOKE_TEST:
    print()
    print('  >>> ĐANG Ở CHẾ ĐỘ SMOKE TEST — artifact này KHÔNG dùng để bàn giao <<<')

## Kết luận

Bản 30 cột đã được đóng gói đầy đủ và nghiệm thu: `model_k30.pkl` nạp lại được bằng
`pickle.load` thuần, dựng vector theo contract cho ra đúng `golden_predictions_k30.csv` tới
từng bit, và bản booster dạng text khớp không sai một chữ số.

Cái nó **không** có, và `metadata_k30.json` ghi rõ chứ không để trống: một con số trên
`rct_holdout`. Bản 69 cột có (`0,0161`, KTC chứa 0, xếp 6/6). Đổi sang bản 30 cột nghĩa là mô
hình chạy production không còn bằng chứng nào trên tập giữ cuối — đó là cái giá của việc giữ
quy tắc "một lần chọn, một lần mở holdout", và là một đánh đổi có ý thức chứ không phải một
chỗ bỏ sót.